# 03 — Feature Engineering

Documents every transformation applied in `src/features.py::prepare_features`.
Run cells top-to-bottom; outputs are evidence for the final report.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sns.set_theme(style='whitegrid')

from src.features import (
    TARGET_COL,
    load_train_test,
    prepare_features,
    split_features_target,
)

train_raw, test_raw = load_train_test(ROOT / 'data')
X_raw, y = split_features_target(train_raw)
print('Raw shape:', X_raw.shape)
X_raw.head(3)

## 1. Columns dropped

| Column | Reason |
|---|---|
| `id` | Opaque row identifier — no signal |
| `LoanNr_ChkDgt` | SBA loan number — identifier |
| `Name` | 37 615 unique business names — no generalisation |
| `BalanceGross` | Single value (`$0.00`) across all rows — zero variance |

In [ ]:
print('BalanceGross unique values:', X_raw['BalanceGross'].unique())
print('Name unique count:', X_raw['Name'].nunique(), '/', len(X_raw))

## 2. Currency parsing — DisbursementGross

In [ ]:
raw_vals = X_raw['DisbursementGross'].head(5).tolist()
print('Before:', raw_vals)

X_eng = prepare_features(X_raw.copy())
print('After:', X_eng['DisbursementGross'].head(5).tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
X_eng['DisbursementGross'].hist(bins=50, ax=axes[0])
axes[0].set_title('DisbursementGross (raw scale)')
X_eng['log_DisbursementGross'].hist(bins=50, ax=axes[1], color='steelblue')
axes[1].set_title('log_DisbursementGross (log1p)')
plt.tight_layout(); plt.show()

## 3. Binary flag columns — RevLineCr & LowDoc

`RevLineCr` has four codes (`Y`, `N`, `0`, `T`) where only `Y` means revolving credit.
`LowDoc` has seven codes where only `Y` means the low-documentation programme was used.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, col in zip(axes, ['RevLineCr', 'LowDoc']):
    raw = X_raw[col].value_counts()
    raw.plot(kind='bar', ax=ax, title=f'{col} — raw codes')
    ax.set_xlabel('')
plt.tight_layout(); plt.show()

print('After encoding:')
for col in ['RevLineCr', 'LowDoc']:
    print(f'  {col}:', X_eng[col].value_counts().to_dict())

## 4. Date features + disbursement lag

In [ ]:
date_cols = ['ApprovalDate_year', 'ApprovalDate_month',
             'DisbursementDate_year', 'DisbursementDate_month',
             'disburse_lag_years', 'is_crisis_period']
X_eng[date_cols].describe()

In [ ]:
# Approval volume by year — shows 2008 dip
approval_counts = X_eng.groupby('ApprovalDate_year').size()
approval_counts.plot(kind='bar', figsize=(12, 3), title='Loans approved by year')
plt.tight_layout(); plt.show()

## 5. Job ratio feature

In [ ]:
print('jobs_per_emp describe:')
print(X_eng['jobs_per_emp'].describe())

fig, ax = plt.subplots(figsize=(8, 3))
X_eng['jobs_per_emp'].clip(upper=5).hist(bins=40, ax=ax)
ax.set_title('jobs_per_emp (clipped at 5 for display)')
plt.tight_layout(); plt.show()

## 6. FranchiseCode → is_franchised

In [ ]:
print('Raw FranchiseCode (top 5):'); print(X_raw['FranchiseCode'].value_counts().head())
print('\nis_franchised (0=no, 1=yes):'); print(X_eng['is_franchised'].value_counts())

# Approval rate by franchise status
pd.crosstab(X_eng['is_franchised'], y, normalize='index').plot(
    kind='bar', stacked=True, title='Accept rate by franchise status')
plt.tight_layout(); plt.show()

## 7. Final feature schema

In [ ]:
print(f'Engineered feature count: {X_eng.shape[1]} (was {X_raw.shape[1]})')
print('\nColumns and dtypes:')
print(X_eng.dtypes.to_string())

## 8. Correlation heatmap (numeric features vs target)

In [ ]:
num_df = X_eng.select_dtypes(include='number').copy()
num_df[TARGET_COL] = y.values

corr = num_df.corr()[TARGET_COL].drop(TARGET_COL).sort_values()
corr.plot(kind='barh', figsize=(7, 5), title=f'Pearson correlation with {TARGET_COL}')
plt.tight_layout(); plt.show()